# Taylor Series · Statics · Dispersion · Exit Codes · Notation

| § | Topic | Key result |
|---|---|---|
|1| `$?` exit codes 0/1/2 | taxonomy, pipeline, Python integration |
|2| Taylor 5% radius | $\cos x\approx1-x^2/2$ valid to $x<0.55$ rad for <5% error |
|3| Dispersion Taylor | $\omega(k)=\omega_0+v_g k+\tfrac12\beta_2 k^2+\cdots$; GVD, chirp, GS |
|4| Statics: object flip | tipping when CoM $x > x_{\rm edge}$; torque $\tau=r\times F$ |
|5| Notation | bra-ket, nabla, tensor, Fourier convention, GS phase notation |


## §1 `$?` Exit-Code Taxonomy: 0 · 1 · 2

| Code | POSIX meaning | Python `sys.exit` | Common cause |
|---|---|---|---|
|**0**| Success | `sys.exit(0)` or fall-off | No error |
|**1**| General error | `sys.exit(1)` | Caught exception, logic fail |
|**2**| Misuse / bad args | `argparse` auto-exits 2 | Wrong CLI flag, bad type |
|**126**| Not executable | — | Permission denied |
|**127**| Command not found | — | PATH miss |
|**128+n**| Fatal signal n | — | SIGSEGV=139, SIGTERM=143 |

**Pipeline propagation**: `cmd1 | cmd2` — `$?` is exit code of *last* command.
Use `${PIPESTATUS[@]}` (bash) or `set -o pipefail` to catch upstream failures.

```bash
python script.py | grep pattern
echo $?          # grep exit code, NOT python exit code!

set -o pipefail
python script.py | grep pattern
echo $?          # now propagates python failure
```


In [ ]:
import subprocess, sys, os

def run_exit(code_val: int) -> int:
    """Spawn subprocess that exits with given code, return captured $?."""
    r = subprocess.run(
        [sys.executable, '-c', f'import sys; sys.exit({code_val})'],
        capture_output=True)
    return r.returncode

print("Exit code capture:")
for ev in [0, 1, 2, 42, 127]:
    actual = run_exit(ev)
    print(f"  sys.exit({ev:>3}) -> $? = {actual}")

# Pipeline $? gotcha
print("\nPipeline $? trap:")
# python fails, grep succeeds on empty -> $?=1 without pipefail
r = subprocess.run(
    'python -c "import sys; sys.exit(1)" | cat; echo $?',
    shell=True, capture_output=True, text=True)
print(f"  Without pipefail: pipe stdout={r.stdout.strip()!r}")

r2 = subprocess.run(
    'set -o pipefail; python -c "import sys; sys.exit(1)" | cat; echo $?',
    shell=True, capture_output=True, text=True, executable='/bin/bash'
    if os.path.exists('/bin/bash') else None)
print(f"  With pipefail:    pipe stdout={r2.stdout.strip()!r}")

# Context-manager pattern that captures $?
import contextlib

class ShellResult:
    def __init__(self): self.code = None
    def __repr__(self): return f"ShellResult($?={self.code})"

@contextlib.contextmanager
def capture_exit():
    r = ShellResult()
    yield r
    # code set by caller

print("\nPython exception -> exit code mapping:")
EXCEPTION_CODES = {
    ValueError:      2,   # bad arg (like argparse)
    FileNotFoundError: 3,
    PermissionError:   4,
    RuntimeError:      1,
    KeyboardInterrupt: 130,  # 128 + SIGINT(2)
}
for exc, code in EXCEPTION_CODES.items():
    print(f"  {exc.__name__:25} -> sys.exit({code})")


## §2 Taylor Series — 5% Accuracy Radius

For $f(x)$ expanded around $x=0$ to order $n$:
$$f(x) \approx \sum_{k=0}^{n} \frac{f^{(k)}(0)}{k!} x^k + O(x^{n+1})$$

**5% accuracy radius** $x_{5\%}$: largest $x$ where $|f(x)-T_n(x)|/|f(x)| < 0.05$

| Function | 1st-order $T_1$ | 2nd-order $T_2$ | $x_{5\%}$ (2nd) |
|---|---|---|---|
| $\sin x$ | $x$ | $x$ | 0.55 rad |
| $\cos x$ | $1$ | $1-x^2/2$ | 0.55 rad |
| $e^x$ | $1+x$ | $1+x+x^2/2$ | 0.42 |
| $(1+x)^{1/2}$ | $1+x/2$ | $1+x/2-x^2/8$ | 0.39 |
| $\ln(1+x)$ | $x$ | $x-x^2/2$ | 0.44 |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

x = np.linspace(-2.0, 2.0, 4000)

functions = {
    r'$\sin x$':       (np.sin(x),    x,           x - x**3/6),
    r'$\cos x$':       (np.cos(x),    np.ones_like(x), 1 - x**2/2),
    r'$e^x$':          (np.exp(x),    1+x,         1+x+x**2/2),
    r'$(1+x)^{1/2}$':  (np.sqrt(np.clip(1+x,0,None)), 1+x/2, 1+x/2-x**2/8),
    r'$\ln(1+x)$':     (np.log(np.clip(1+x,1e-9,None)), x, x-x**2/2),
}

# Find 5% accuracy radius for 2nd-order Taylor
print(f"{'Function':>20}  {'x_5% (T2)':>12}  {'x_5% (T1)':>12}")
for name, (f_true, T1, T2) in functions.items():
    denom = np.abs(f_true) + 1e-10
    err1  = np.abs(f_true - T1) / denom
    err2  = np.abs(f_true - T2) / denom
    xpos  = x[x > 0]
    e1p   = err1[x > 0]; e2p = err2[x > 0]
    x5_1  = xpos[e1p < 0.05].max() if np.any(e1p < 0.05) else 0.0
    x5_2  = xpos[e2p < 0.05].max() if np.any(e2p < 0.05) else 0.0
    print(f"{name:>20}  {x5_2:>12.3f}  {x5_1:>12.3f}")

# ── Physics: pendulum small angle ─────────────────────────────────────────────
print("\nPendulum: T = 2π√(L/g) × correction")
print("  T(θ) = T₀(1 + θ²/16 + 11θ⁴/3072 + ...)")
theta_deg = np.array([5, 10, 15, 20, 30, 45, 60])
theta_rad = np.radians(theta_deg)
# Exact: T/T0 from elliptic integral (series)
T_corr_exact = 1 + theta_rad**2/16 + 11*theta_rad**4/3072
T_small = 1 + theta_rad**2/16   # first correction only
for th_d, exact, sm in zip(theta_deg, T_corr_exact, T_small):
    err_pct = abs(exact - sm)/exact * 100
    flag = " ← > 5% err" if err_pct > 5 else ""
    print(f"  {th_d:>3}°: T/T₀ = {exact:.5f}  small-angle = {sm:.5f}  "
          f"err = {err_pct:.2f}%{flag}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10), facecolor='#0d1117')
gs  = GridSpec(2, 3, fig, hspace=0.48, wspace=0.34)

colors = ['#4fc3f7','#ffa726','#66bb6a','#ef5350','#b39ddb']
x_plot = np.linspace(-1.5, 1.5, 1000)

for idx, (name, (f_t, T1v, T2v)) in enumerate(functions.items()):
    ax = fig.add_subplot(gs[idx//3, idx%3]); ax.set_facecolor('#161b22')
    mask = (x >= -1.5) & (x <= 1.5)
    ax.plot(x[mask], f_t[mask], '#e8e8e8', lw=2.2, label='Exact')
    ax.plot(x[mask], T1v[mask] if not np.all(np.isnan(T1v[mask])) else np.zeros_like(x[mask]),
            '#ffa726', lw=1.5, ls='--', label='$T_1$')
    ax.plot(x[mask], T2v[mask], '#4fc3f7', lw=1.5, ls=':', label='$T_2$')
    # Shade 5% error band
    denom_m = np.abs(f_t[mask]) + 1e-10
    err2_m  = np.abs(f_t[mask] - T2v[mask]) / denom_m
    x_m = x[mask]
    ax.fill_between(x_m, f_t[mask]-0.05*np.abs(f_t[mask]),
                    f_t[mask]+0.05*np.abs(f_t[mask]), alpha=0.12, color='white')
    ax.set_title(name, color='#e8e8e8', fontweight='bold')
    ax.legend(fontsize=7); ax.grid(alpha=0.2)
    ax.tick_params(colors='#aaa')
    [sp.set_color('#333') for sp in ax.spines.values()]
    ax.set_xlim(-1.5,1.5)

# Panel 6: error vs x for all functions
ax_last = fig.add_subplot(gs[1, 2]); ax_last.set_facecolor('#161b22')
mask = (x >= 0) & (x <= 1.5)
for idx, (name, (f_t, T1v, T2v)) in enumerate(functions.items()):
    denom_m = np.abs(f_t[mask]) + 1e-10
    err2_m  = np.abs(f_t[mask] - T2v[mask]) / denom_m * 100
    ax_last.plot(x[mask], err2_m, color=colors[idx], lw=1.8, label=name)
ax_last.axhline(5, color='white', ls='--', lw=1.2, label='5% threshold')
ax_last.set_xlabel('x'); ax_last.set_ylabel('Relative error (%)')
ax_last.set_title('$T_2$ relative error vs $x$', color='#e8e8e8', fontweight='bold')
ax_last.legend(fontsize=6); ax_last.grid(alpha=0.2); ax_last.set_ylim(0, 25)
ax_last.tick_params(colors='#aaa'); [sp.set_color('#333') for sp in ax_last.spines.values()]

plt.suptitle('§2  Taylor Series — 5% Accuracy Radius — Physics Truncation Map',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s2_taylor.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §3 Dispersion Relation Taylor Expansion

**General dispersion** $\omega(k)$ expanded around carrier $k_0$:
$$\omega(k) = \omega_0 + \underbrace{\left.\frac{d\omega}{dk}\right|_{k_0}}_{v_g}(k-k_0)
  + \frac{1}{2}\underbrace{\left.\frac{d^2\omega}{dk^2}\right|_{k_0}}_{\beta_2}(k-k_0)^2
  + \frac{1}{6}\beta_3(k-k_0)^3 + \cdots$$

| Term | Name | Effect |
|---|---|---|
|$v_g = d\omega/dk$| Group velocity | Envelope speed |
|$\beta_2 = d^2\omega/dk^2$| GVD (Group Velocity Dispersion) | Pulse broadening, chirp |
|$\beta_3$| Third-order dispersion | Asymmetric broadening |

**GS phase recovery** uses dispersion: the diversity $D$ is exactly $\beta_2 L$ in SI units. Near-zero $\beta_2$ (anomalous↔normal boundary) gives $|D|\approx 0$ — no diversity, GS fails.

**Modern physics examples**:
- Photon: $\omega=ck$, $v_g=c$, $\beta_2=0$ (linear, dispersion-free)
- Phonon (acoustic): $\omega=v_s k$ for $ka\ll1$, bends at BZ edge
- Relativistic massive: $\omega=c\sqrt{k^2+m^2c^2/\hbar^2}$, $v_g=\hbar k c^2/E < c$
- Gravity wave (deep water): $\omega=\sqrt{gk}$, $v_g=\frac{1}{2}\sqrt{g/k}$


In [ ]:
from scipy.constants import c as c_light, hbar, m_e

fig2, axes2 = plt.subplots(2, 3, figsize=(16, 9), facecolor='#0d1117')
for ax in axes2.flat: ax.set_facecolor('#161b22')

k = np.linspace(0.01, 4.0, 2000)   # normalised wavenumber

# 1. Photon: ω = ck (linear)
omega_photon = k   # normalised
vg_photon    = np.gradient(omega_photon, k)
ax = axes2[0,0]
ax.plot(k, omega_photon, '#4fc3f7', lw=2.2, label=r'$\omega=ck$')
ax.set_title('Photon: linear, $v_g=c$', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# 2. Gravity wave: ω = sqrt(gk)
g  = 9.81
omega_grav = np.sqrt(k)   # normalised g=1
vg_grav    = np.gradient(omega_grav, k)
ax = axes2[0,1]
ax.plot(k, omega_grav, '#ffa726', lw=2.2, label=r'$\omega=\sqrt{gk}$')
ax.plot(k, k, '#555', ls='--', lw=1, label='linear ref')
ax.set_title(r'Deep water: $v_g=\frac{1}{2}v_\phi$', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# 3. Relativistic massive particle
# ω = sqrt(k² + 1)  (mc/ℏ = 1 normalised)
omega_rel = np.sqrt(k**2 + 1.0)
vg_rel    = np.gradient(omega_rel, k)
ax = axes2[0,2]
ax.plot(k, omega_rel, '#66bb6a', lw=2.2, label=r'$\omega=\sqrt{k^2+1}$')
ax.plot(k, k, '#555', ls='--', lw=1, label='massless limit')
ax.plot(k, np.ones_like(k), '#ef5350', ls=':', lw=1, label='$m c^2/\hbar$')
ax.set_title(r'Massive: $v_g=k/\omega < c$', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# 4. Optical fiber dispersion: β₂ profile
# Typical SMF-28: β₂ = -21.7 ps²/km at 1550nm, zero crossing at ~1310nm
lambda_nm = np.linspace(1200, 1700, 1000)
# D_chrom ~ A*(lambda - lambda_ZD)  [ps/nm/km]
lambda_ZD = 1310.0   # nm, zero dispersion wavelength
D_chrom   = 0.092 * (lambda_nm - lambda_ZD)   # ps/nm/km
beta2     = -lambda_nm**2 / (2*np.pi*3e8*1e-9) * D_chrom * 1e-3  # simplified units

ax = axes2[1,0]
ax.plot(lambda_nm, D_chrom, '#b39ddb', lw=2.2, label='$D$ (ps/nm/km)')
ax.axhline(0, color='white', lw=1.2, ls='--')
ax.axvline(1310, color='#ffa726', lw=1, ls=':', label='ZDW 1310nm')
ax.axvline(1550, color='#4fc3f7', lw=1, ls=':', label='C-band 1550nm')
ax.fill_between(lambda_nm, 0, D_chrom, where=D_chrom>0, alpha=0.15, color='#ef5350',
                label='Anomalous')
ax.fill_between(lambda_nm, 0, D_chrom, where=D_chrom<0, alpha=0.15, color='#4fc3f7',
                label='Normal')
ax.set_xlabel('λ (nm)'); ax.set_ylabel('D (ps/nm/km)')
ax.set_title('Fiber dispersion — GS diversity $D\\propto\\beta_2 L$',
             color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# 5. GVD impact on GS: diversity vs. beta2*L
beta2_vals = np.linspace(-200, 200, 800)   # ps²
corr_sim = np.exp(-0.5*(beta2_vals/50)**2)   # correlation ~ exp(-D²/2σ²) toy model
ax = axes2[1,1]
ax.plot(beta2_vals, corr_sim, '#ef5350', lw=2.2)
ax.axhline(0.95, color='#ffa726', ls='--', lw=1.5, label='corr=0.95 (GS fails)')
ax.axvline(-5000, color='#555', ls=':', lw=0.8)
for D_min in [-5000, 5000]:
    ax.axvline(D_min/1000, color='#66bb6a', ls=':', lw=1.2)
ax.fill_between(beta2_vals, corr_sim, 0.95, where=corr_sim>0.95,
                alpha=0.25, color='#ef5350', label='GS failure zone')
ax.set_xlabel(r'$\beta_2 L$ (ps²)'); ax.set_ylabel('Input correlation')
ax.set_title('GS diversity: |D|≥5000 ps² needed\n(D=-600 → corr>0.95, fails)',
             color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# 6. Taylor orders for cos(x) accuracy — dispersion analogy
k_taylor = np.linspace(0, 3.0, 1000)
omega_exact = np.cos(k_taylor - np.pi/2)   # sin(k) as dispersion proxy
T1 = k_taylor
T2 = k_taylor - k_taylor**3/6
T3 = k_taylor - k_taylor**3/6 + k_taylor**5/120
ax = axes2[1,2]
ax.plot(k_taylor, omega_exact, '#e8e8e8', lw=2.2, label='Exact $\sin k$')
ax.plot(k_taylor, T1, '#ffa726', lw=1.6, ls='--', label='$k$ (1st)')
ax.plot(k_taylor, T2, '#4fc3f7', lw=1.6, ls='--', label='$k-k^3/6$ (3rd)')
ax.plot(k_taylor, T3, '#66bb6a', lw=1.6, ls='--', label='5th order')
ax.fill_between(k_taylor, T2, omega_exact, alpha=0.15, color='#ef5350', label='GVD error')
ax.set_xlabel('k'); ax.set_ylabel(r'$\omega(k)$')
ax.set_title('Dispersion Taylor orders\n(error = GVD contribution)', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.2); ax.set_ylim(-0.2, 3.5)

for ax in axes2.flat:
    ax.tick_params(colors='#aaa')
    [sp.set_color('#333') for sp in ax.spines.values()]
    ax.xaxis.label.set_color('#ccc') if ax.get_xlabel() else None
    ax.yaxis.label.set_color('#ccc') if ax.get_ylabel() else None
    ax.title.set_color('#e8e8e8')

plt.suptitle(r'§3  Dispersion Taylor: $\omega(k)=\omega_0+v_g k+\frac{1}{2}\beta_2 k^2+\cdots$',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s3_dispersion.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §4 Statics — Object Tipping / Flipping

**Tipping condition**: object tips when the CoM projects *outside* the support base.

For a block of height $h$, width $w$, mass $m$ on a horizontal surface:
- Applied horizontal force $F$ at height $h_F$
- **Tip** when: $F \cdot h_F > mg \cdot (w/2)$, i.e., overturning moment > restoring moment
- **Tipping angle** (no force): tilts until CoM over edge → $\theta_{\rm tip} = \arctan(w/h)$

**Stability margin**: $\Delta x = x_{\rm edge} - x_{\rm CoM}$ (positive = stable)

**Dynamic flip** (sudden impulse $J$ at $h_F$):
$$\omega_{\rm tip} = \frac{J \cdot h_F}{I_{\rm edge}}, \qquad I_{\rm edge} = I_{\rm cm} + m d^2$$


In [ ]:
from scipy.integrate import solve_ivp

fig3, axes3 = plt.subplots(2, 3, figsize=(16, 9), facecolor='#0d1117')
for ax in axes3.flat: ax.set_facecolor('#161b22')

g_stat = 9.81

# ── Block geometry ────────────────────────────────────────────────────────────
w = 0.4   # m width
h = 1.0   # m height
m = 10.0  # kg

# Tipping angle
theta_tip = np.arctan(w/h)
print(f"Block {w}m × {h}m, m={m}kg")
print(f"  Tipping angle θ_tip = {np.degrees(theta_tip):.2f}°")
print(f"  (tan θ = w/h = {w/h:.3f})")

# Critical force at height h_F
h_F_vals = np.linspace(0.1, h, 100)
F_crit   = m * g_stat * (w/2) / h_F_vals
print(f"\nCritical tipping force F_crit at various heights:")
for hf, Fc in zip([0.1, 0.3, 0.5, h/2, h], F_crit[[0,20,39,49,99]]):
    print(f"  h_F = {hf:.2f}m:  F_crit = {Fc:.1f} N")

# ── Stability map: F vs h_F phase diagram ─────────────────────────────────────
ax = axes3[0,0]
h_F_grid = np.linspace(0.05, h, 200)
F_crit_grid = m * g_stat * (w/2) / h_F_grid
ax.plot(h_F_grid, F_crit_grid, '#ef5350', lw=2.5, label='Tipping boundary')
ax.fill_between(h_F_grid, F_crit_grid, F_crit_grid.max(),
                alpha=0.2, color='#ef5350', label='Flips')
ax.fill_between(h_F_grid, 0, F_crit_grid,
                alpha=0.2, color='#66bb6a', label='Stable')
ax.set_xlabel('Force height $h_F$ (m)')
ax.set_ylabel('Applied force $F$ (N)')
ax.set_title('Tipping stability map\n$F\\cdot h_F = mg\\cdot(w/2)$', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.set_ylim(0, 500)

# ── Block diagram at various tilt angles ─────────────────────────────────────
ax = axes3[0,1]
theta_vals = np.linspace(0, theta_tip + 0.3, 6)
colors_t = plt.cm.RdYlGn_r(np.linspace(0, 1, len(theta_vals)))
for theta_v, col in zip(theta_vals, colors_t):
    # Block corners relative to pivot (bottom-right corner)
    pivot = np.array([0.0, 0.0])
    corners_local = np.array([
        [-w, 0], [0, 0], [0, h], [-w, h], [-w, 0]
    ])
    R = np.array([[np.cos(theta_v), -np.sin(theta_v)],
                  [np.sin(theta_v),  np.cos(theta_v)]])
    corners_rot = (R @ corners_local.T).T
    # CoM position
    com_local = np.array([-w/2, h/2])
    com_rot   = R @ com_local
    ax.plot(corners_rot[:,0], corners_rot[:,1], color=col, lw=1.5)
    ax.plot(com_rot[0], com_rot[1], 'o', color=col, ms=5)
    if theta_v > theta_tip:
        ax.annotate('TIPPED', (com_rot[0], com_rot[1]), color='#ef5350',
                    fontsize=7, xytext=(0.1, 0.5), arrowprops=dict(arrowstyle='->', color='#ef5350'))
ax.axhline(0, color='#555', lw=1); ax.axvline(0, color='#ffa726', lw=1.5, ls='--', label='Pivot')
ax.set_aspect('equal'); ax.set_xlim(-1.2, 0.8); ax.set_ylim(-0.3, 1.4)
ax.set_title(f'Block tipping at θ_tip={np.degrees(theta_tip):.1f}°', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# ── Rotational dynamics after tip ─────────────────────────────────────────────
# I_edge = m(w²+h²)/3  (uniform rectangle pivoting at corner)
I_edge = m * (w**2 + h**2) / 3
# ODE: I θ'' = m*g*(com_x - 0) where com_x = distance from pivot along ground
# = (w/2)cosθ - (h/2)sinθ
def tip_ode(t, y):
    theta_o, omega_o = y
    # Torque about pivot (bottom-right corner, positive = tipping)
    # com from pivot in rotated frame: (-w/2, h/2) → world x = -w/2*cos + h/2*sin...
    # Net torque = m*g*x_com where x_com = -(w/2)cos(θ)+( h/2)sin(θ) wait let's be careful
    # At angle θ (tilted from vertical), pivot at origin, block tilted θ from upright
    # CoM from pivot: x_com = (h/2)sin(θ) - (w/2)cos(θ)
    # Gravity torque = m*g*x_com (positive when tipping CW)
    x_com = (h/2)*np.sin(theta_o) - (w/2)*np.cos(theta_o)
    tau   = m * g_stat * x_com
    return [omega_o, tau / I_edge]

# Initial condition: just at tipping angle with tiny perturbation
theta0  = theta_tip + 0.01
omega0  = 0.0
t_span  = (0, 2.0)
t_eval  = np.linspace(0, 2.0, 500)
sol = solve_ivp(tip_ode, t_span, [theta0, omega0], t_eval=t_eval,
                max_step=0.01, method='RK45')

# Find when θ = π/2 (block hits ground)
hit_idx = np.argmax(sol.y[0] >= np.pi/2) if np.any(sol.y[0] >= np.pi/2) else -1
t_hit   = sol.t[hit_idx] if hit_idx > 0 else None
print(f"\nRotational fall after tipping:")
print(f"  I_edge = {I_edge:.4f} kg·m²")
print(f"  Time to hit ground: {t_hit:.3f}s" if t_hit else "  Didn't reach ground in 2s")

ax = axes3[0,2]
ax.plot(sol.t, np.degrees(sol.y[0]), '#4fc3f7', lw=2.2, label='θ(t)')
ax.axhline(90, color='#ef5350', ls='--', lw=1.5, label='Ground (90°)')
ax.axhline(np.degrees(theta_tip), color='#ffa726', ls=':', lw=1.2, label='θ_tip')
if t_hit: ax.axvline(t_hit, color='#66bb6a', ls=':', lw=1.2, label=f't_hit={t_hit:.2f}s')
ax.set_xlabel('t (s)'); ax.set_ylabel('θ (°)')
ax.set_title('Rotational fall after tipping', color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# ── w/h ratio vs. tipping angle ───────────────────────────────────────────────
ax = axes3[1,0]
wh_ratios = np.linspace(0.1, 3.0, 200)
tip_angles = np.degrees(np.arctan(wh_ratios))
ax.plot(wh_ratios, tip_angles, '#b39ddb', lw=2.5)
ax.axhline(45, color='#ffa726', ls='--', lw=1.2, label='45° (w=h cube)')
ax.fill_between(wh_ratios, 0, tip_angles, alpha=0.15, color='#66bb6a', label='Stable zone')
ax.set_xlabel('w/h aspect ratio'); ax.set_ylabel('Tipping angle (°)')
ax.set_title('θ_tip vs aspect ratio\n(tall → tips easily, wide → stable)',
             color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
ax.plot(w/h, np.degrees(theta_tip), 'r*', ms=14, label=f'Our block ({w/h:.2f})')

# ── Torque diagram ─────────────────────────────────────────────────────────────
ax = axes3[1,1]; ax.set_aspect('equal'); ax.set_xlim(-1, 2.5); ax.set_ylim(-0.5, 1.5)
# Draw block upright
rect_x = [0, w, w, 0, 0]; rect_y = [0, 0, h, h, 0]
ax.fill(rect_x, rect_y, alpha=0.3, color='#4fc3f7')
ax.plot(rect_x, rect_y, '#4fc3f7', lw=2)
# CoM
ax.plot(w/2, h/2, 'o', color='#ffa726', ms=12, label='CoM')
# Gravity arrow
ax.annotate('', (w/2, h/2-0.3), (w/2, h/2),
            arrowprops=dict(arrowstyle='->', color='#ef5350', lw=2))
ax.text(w/2+0.05, h/2-0.15, '$mg$', color='#ef5350', fontsize=11)
# Applied force
h_F_demo = 0.7
ax.annotate('', (w+0.4, h_F_demo), (w, h_F_demo),
            arrowprops=dict(arrowstyle='<-', color='#66bb6a', lw=2))
ax.text(w+0.42, h_F_demo+0.03, '$F$', color='#66bb6a', fontsize=11)
# Pivot
ax.plot(w, 0, 's', color='#ffa726', ms=12, label='Pivot')
# Moment arms
ax.annotate('', (w, 0), (w/2, 0),
            arrowprops=dict(arrowstyle='<->', color='white', lw=1.5))
ax.text(w/2+0.05, -0.12, '$w/2$', color='white', fontsize=9)
ax.annotate('', (w+0.05, 0), (w+0.05, h_F_demo),
            arrowprops=dict(arrowstyle='<->', color='#aaa', lw=1.5))
ax.text(w+0.1, h_F_demo/2, '$h_F$', color='#aaa', fontsize=9)
ax.set_title('Torque diagram\n$\\tau_{tip}=F h_F$,  $\\tau_{rest}=mg(w/2)$',
             color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.15)
ax.axhline(0, color='#555', lw=1)

# ── Stability margin for moving CoM ───────────────────────────────────────────
ax = axes3[1,2]
# Object on incline at angle α: stability margin = (w/2)cosα - (h/2)sinα
alpha_deg = np.linspace(0, 90, 200)
alpha_rad = np.radians(alpha_deg)
margin_w04 = (0.4/2)*np.cos(alpha_rad) - (1.0/2)*np.sin(alpha_rad)
margin_w08 = (0.8/2)*np.cos(alpha_rad) - (1.0/2)*np.sin(alpha_rad)
margin_w02 = (0.2/2)*np.cos(alpha_rad) - (1.0/2)*np.sin(alpha_rad)
ax.plot(alpha_deg, margin_w04, '#4fc3f7', lw=2, label='w=0.4m (our block)')
ax.plot(alpha_deg, margin_w08, '#66bb6a', lw=2, label='w=0.8m (wide)')
ax.plot(alpha_deg, margin_w02, '#ef5350', lw=2, label='w=0.2m (tall/narrow)')
ax.axhline(0, color='white', ls='--', lw=1.5)
ax.fill_between(alpha_deg, 0, 0.5, alpha=0.1, color='#66bb6a')
ax.fill_between(alpha_deg, -0.5, 0, alpha=0.1, color='#ef5350')
ax.set_xlabel('Incline angle α (°)'); ax.set_ylabel('Stability margin (m)')
ax.set_title('Stability margin on incline\n(zero crossing = tipping angle)',
             color='#e8e8e8', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.2); ax.set_ylim(-0.5, 0.5)

for ax in axes3.flat:
    ax.tick_params(colors='#aaa')
    [sp.set_color('#333') for sp in ax.spines.values()]
    ax.xaxis.label.set_color('#ccc') if ax.get_xlabel() else None
    ax.yaxis.label.set_color('#ccc') if ax.get_ylabel() else None
    ax.title.set_color('#e8e8e8')

plt.suptitle('§4  Statics: Object Tipping — Torque Balance — Rotational Fall',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s4_statics.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## §5 Notation — Physics LaTeX Standard

The "going nowhere notation" problem: inconsistent symbols across courses destroy comprehension.
This section establishes canonical notation for all topics in this repo.

---

### Fourier Transform — Two Conventions (pick one, stick to it)

| Convention | Definition | Used in |
|---|---|---|
| **Physics** (ω) | $\tilde{f}(\omega)=\int f(t)e^{-i\omega t}dt$ | QM, optics |
| **Engineering** (f) | $F(f)=\int f(t)e^{-i2\pi ft}dt$ | DSP, comms |
| **GS this repo** | $\hat{u}(k)=\int u(x)e^{-ikx}dx$ | spatial, phase retrieval |

**Rule**: always write the exponent sign convention explicitly in the caption.

---

### Bra-Ket (Dirac) Notation

$$|\psi\rangle \text{ (ket)}, \quad \langle\phi| \text{ (bra)}, \quad
\langle\phi|\psi\rangle = \int\phi^*(x)\psi(x)\,dx$$

$$\hat{A}|\psi\rangle = a|\psi\rangle \quad\text{(eigenvalue eq)}, \qquad
[\hat{x},\hat{p}] = i\hbar$$

---

### Vector Calculus — ∇ Operations

| Op | Cartesian | Physics meaning |
|---|---|---|
| $\nabla f$ | $(\partial_x f, \partial_y f, \partial_z f)$ | Gradient: direction of steepest ascent |
| $\nabla\cdot\mathbf{F}$ | $\partial_x F_x + \partial_y F_y + \partial_z F_z$ | Divergence: source/sink density |
| $\nabla\times\mathbf{F}$ | $(\partial_y F_z-\partial_z F_y,\ldots)$ | Curl: rotation / circulation |
| $\nabla^2 f$ | $\partial_{xx}f+\partial_{yy}f+\partial_{zz}f$ | Laplacian: diffusion operator |

---

### Tensor Index Notation (Einstein summation)

$$T^{\mu\nu}F_{\nu\rho} \equiv \sum_\nu T^{\mu\nu}F_{\nu\rho}$$

- **Upper** (contravariant): transforms like $\partial/\partial x_\mu$
- **Lower** (covariant): transforms like $\partial/\partial x^\mu$
- Metric raises/lowers: $v_\mu = g_{\mu\nu}v^\nu$

---

### GS Phase Retrieval — Canonical Notation (this repo)

| Symbol | Meaning |
|---|---|
| $u(x)$ | Input field (complex amplitude) |
| $U(k) = \mathcal{F}[u]$ | Spatial spectrum |
| $D$ | Dispersion diversity (ps² or rad/m) — **must** have $|D|\geq5000$ |
| $\phi(x)$ | Recovered phase |
| $\rho(x) = |u(x)|$ | Measured amplitude (constraint) |
| $n_{\rm iter}$ | GS iteration count — use $\geq50$ |
| $\hat{u}_D(k)$ | Dispersed spectrum: $U(k)e^{i D k^2/2}$ |


In [ ]:
# ── Notation verification: show both FT conventions give same physics ──────────
import numpy as np
import matplotlib.pyplot as plt

# Gaussian pulse in both conventions
t = np.linspace(-5, 5, 4096)
dt = t[1]-t[0]
sigma_t = 1.0
f_t = np.exp(-t**2 / (2*sigma_t**2))

# Physics convention: omega
N = len(t)
omega_phys = 2*np.pi * np.fft.fftfreq(N, d=dt)
F_phys = np.fft.fft(f_t) * dt   # ~int f(t) e^{-i omega t} dt

# Engineering convention: frequency f
freq_eng = np.fft.fftfreq(N, d=dt)
F_eng = np.fft.fft(f_t) * dt

# Relation: F_eng(f) = F_phys(ω=2πf)
# Both should give Gaussian with σ_ω = 1/σ_t (physics) or σ_f = 1/(2πσ_t) (eng)
sigma_omega_expected = 1.0 / sigma_t
sigma_f_expected     = 1.0 / (2*np.pi*sigma_t)

omega_pos = omega_phys[omega_phys > 0]
F_pos     = np.abs(F_phys)[omega_phys > 0]
# Find sigma numerically
peak      = F_pos.max()
half_max  = np.where(F_pos > peak/np.e)
if len(half_max[0]) > 0:
    sigma_meas = (omega_pos[half_max[0][-1]] - omega_pos[half_max[0][0]]) / 2
else:
    sigma_meas = 0
print(f"Fourier convention check:")
print(f"  σ_t = {sigma_t:.2f}  →  σ_ω expected = {sigma_omega_expected:.3f} rad/s")
print(f"  Measured σ_ω ≈ {sigma_meas:.3f} rad/s")
print(f"  σ_f = σ_ω/2π = {sigma_omega_expected/(2*np.pi):.4f} Hz")
print(f"  Uncertainty principle: σ_t × σ_ω = {sigma_t * sigma_omega_expected:.3f} ≥ 0.5 ✓")

# ── GS notation table rendered as matplotlib figure ──────────────────────────
fig4, axes4 = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0d1117')

ax = axes4[0]; ax.set_facecolor('#0d1117'); ax.axis('off')
notation_rows = [
    [r'$u(x)$',            'Input field (complex)'],
    [r'$U(k)=\mathcal{F}[u]$', 'Spatial spectrum'],
    [r'$|D| \geq 5000$',   'Diversity threshold (GS)'],
    [r'$n_{\rm iter}\geq 50$', 'Iteration count'],
    [r'$\phi(x)$',         'Recovered phase'],
    [r'$\rho(x)=|u(x)|$',  'Measured amplitude'],
    [r'$\hat{u}_D(k)=U(k)e^{iDk^2/2}$', 'Dispersed spectrum'],
    [r'$\sigma_t\cdot\sigma_\omega \geq \tfrac{1}{2}$', 'Uncertainty principle'],
    [r'$\omega(k)=\omega_0+v_g k+\tfrac12\beta_2 k^2$', 'Dispersion Taylor'],
    [r'$|\psi\rangle,\,\langle\phi|$', 'Bra-ket (Dirac)'],
    [r'$[\hat{x},\hat{p}]=i\hbar$', 'Canonical commutation'],
    [r'$\nabla^2\phi-\tfrac{1}{c^2}\partial_{tt}\phi=0$', 'Wave equation'],
]
table_data = [[r[0], r[1]] for r in notation_rows]
col_labels = ['Symbol', 'Meaning']
the_table = ax.table(cellText=table_data, colLabels=col_labels,
                     loc='center', cellLoc='left')
the_table.auto_set_font_size(False); the_table.set_fontsize(9)
the_table.scale(1.8, 1.6)
for (i,j), cell in the_table.get_celld().items():
    cell.set_facecolor('#1e2430' if i%2==0 else '#161b22')
    cell.set_text_props(color='#e8e8e8')
    cell.set_edgecolor('#333')
    if i == 0:
        cell.set_facecolor('#0d2040')
        cell.set_text_props(color='#4fc3f7', fontweight='bold')
ax.set_title('GS/Physics Canonical Notation', color='#e8e8e8', fontweight='bold', pad=20)

# FT convention comparison
ax = axes4[1]; ax.set_facecolor('#161b22')
freq_plot = np.fft.fftshift(freq_eng)
F_plot    = np.abs(np.fft.fftshift(F_eng))
omega_plot = np.fft.fftshift(omega_phys)
ax.plot(omega_plot/(2*np.pi), F_plot, '#4fc3f7', lw=2.2, label=r'Eng: $F(f)$ vs $f$')
ax2b = ax.twiny()
ax2b.plot(omega_plot, F_plot, '#ffa726', lw=1.5, ls='--', alpha=0.7,
          label=r'Phys: $\tilde{f}(\omega)$ vs $\omega$')
ax2b.set_xlabel(r'ω (rad/s)', color='#ffa726')
ax2b.tick_params(colors='#ffa726', axis='x')
ax.set_xlabel('f (Hz)', color='#4fc3f7')
ax.set_ylabel('|F| (normalised)')
ax.set_title(r'Both conventions: same $e^{-k^2\sigma^2/2}$ shape'
             '\n(x-axis scale differs by $2\pi$)', color='#e8e8e8', fontweight='bold')
ax.set_xlim(-1.5/(2*np.pi), 1.5/(2*np.pi))
ax2b.set_xlim(-1.5, 1.5)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)
ax.grid(alpha=0.2)
ax.tick_params(colors='#aaa'); [sp.set_color('#333') for sp in ax.spines.values()]

plt.suptitle('§5  Notation: Canonical Physics/GS Symbols + FT Convention Comparison',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/s5_notation.png', dpi=110, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("\n── NOTATION RULES ───────────────────────────────────────────────────────")
print("  FT: always specify exponent sign convention in caption")
print("  GS: D in ps² (fiber) or m²/rad (free-space), |D|≥5000 always")
print("  Index: up=contravariant, down=covariant, repeated=summed")
print("  Operators: hat (^) for quantum, bold for vectors, italic for scalars")


## Summary

| § | Rule / Result |
|---|---|
|**$? codes**| 0=OK, 1=error, 2=bad-args; use `set -o pipefail` to propagate through pipes |
|**Taylor 5%**| $T_2$ (2nd-order) accurate to 5% for $x < 0.55$ rad for sin/cos; pendulum valid to ~15° |
|**Dispersion**| $\omega(k)=\omega_0+v_g k+\frac{1}{2}\beta_2 k^2$; GS needs $|D|=|\beta_2 L|\geq5000$ ps²; zero at ZDW |
|**Statics**| Tip when $F h_F > mg(w/2)$; $\theta_{\rm tip}=\arctan(w/h)$; wide objects survive steep inclines |
|**Notation**| Pick one FT convention, label it; GS canonical: $u,U,D,\phi,\rho,n_{\rm iter}$ |
